
- 18 problem types: BOM, mixed line endings, split rows, delimiter swaps, duplicate headers, ragged rows, UTF-16 splice, 25+ null spellings, symbol noise, homoglyphs, mojibake, bad dates, inconsistent casing, every numeric format, broken JSON, conflicting duplicates

In [3]:
import pandas as pd
import numpy as np
import re, json, ast, csv

pd.set_option("display.max_colwidth", 60)

RAW_FILE = "confusion_financial_transactions.csv"
CLEAN_FILE = "clean_confusion_financial_transactions.csv"


In [4]:
try:
    df_naive = pd.read_csv(RAW_FILE)
    print("Loaded fine:", df_naive.shape)
except Exception as e:
    print(f"{type(e).__name__}: {e}")


UnicodeDecodeError: 'utf-8' codec can't decode byte 0x92 in position 24908: invalid start byte


In [5]:
raw_preview = pd.read_csv(
    RAW_FILE,
    encoding="utf-8",
    encoding_errors="replace",   
    on_bad_lines="skip",         
    engine="python",
)
print(raw_preview.shape)
raw_preview.head(50)


(995, 12)


,Unnamed: 0,txn_id,txn_date,merchant,category,amount,amount.1,currency,status,card_last4,metadata,notes
0,37,TXN50456,20.02.2023,1=1 OR '1'='1,NaN,"94.377,84",(null),usd,[Refunded],8942,"{channel: ""phone"", fee: 3.07, ""loyalty_pts"": 303, ""flagg...",Recurring payment
1,725,TXN50675,10 Apr 2021,Apple Store,// Healthcare//,(262519.00),"347,945.530000",EUR,failed,7179,"{""channel"": ""phone"", ""fee"": 0.27, ""loyalty_pts"": 371, ""f...",NaN
2,785,txn_50608,2026-02-30,Target,Transport,497759.78,NaN,GBP,[Failed],8549,"{""channel"": ""mobile"", ""fee"": 0.44, ""loyalty_pts"": 425, ""...",customer disputed charge
3,822,TXN-50583,12/26/2022,Airbnb,Subscription!!,"36,942.720000",\N,€,NaN,2294,"{channel: ""online"", fee: 0.82, ""loyalty_pts"": 155, ""flag...",customer disputed charge
4,607,txn_50310,09/10/2024,Spotify,SUBSCRIPTION,733902.31◆,#NAME?,GBP,PENDING,9537,"{""channel"": ""in-store"",\t""fee"": 2.64,\t""loyalty_pts"": 3,...",Zü»rich
5,329,TXN50376,4 Jun 2025,Uber,DINING,"393,860.20 USD",4.417E+05 %%,Eur,Completed,NaN,"{""channel"": ""mobile"", ""fee"": 1.64, ""loyalty_pts"": 140, ""...","call○back○re:○acct○#88213,○ref:○'urgent!!'"
6,402,txn_50150≈,2021-03-09T22:09:00Z,Target ‚Äî,Transport,3.733E+05,NaN,€,Refunded,9059,"{""channel"": 'in-store', 'fee': 1.76, 'loyalty_pts': 460,...",аmazon
7,868,TXN50679,Infinity,Chipotle,transport,◊-0◊,"461,315.63 USD",Eur,-Infinity,6227,"{'channel': 'mobile', 'fee': 4.44, 'loyalty_pts': 345, '...","call◆back◆re:◆acct◆#88213,◆ref:◆'urgent!!'"
8,151,TXN-50468,16.01.2020,Uber,Healthcare,NaN,766084.31,$,[Refunded],2231,"{""channel"": ""online"", ""fee"": 4.85, ""loyalty_pts"": 84, ""f...","call back re: acct #88213, ref: 'urgent!!'"
9,769,TXN50213,'2024-08-21,Target,Healthcare\t,6.188E+05,"810,496.160000",Eur,PENDING,6758,"{""channel"": ""mobile"",\t""fee"": 0.72,\t""loyalty_pts"": 238,...",NaN


In [6]:
with open(RAW_FILE, "rb") as f:
    raw = f.read()

had_bom = raw.startswith(b"\xef\xbb\xbf")
if had_bom:
    raw = raw[3:]

text = raw.decode("utf-8", errors="replace")
print(f"BOM present: {had_bom}")
print(f"Decoded length: {len(text):,} characters")


BOM present: True
Decoded length: 305,761 characters


In [7]:
text = text.replace("\r\n", "\n").replace("\r", "\n")


In [ ]:
raw_lines = text.split("\n")
row_start = re.compile(r"^\d+,")

fixed_lines = []
for line in raw_lines:
    if row_start.match(line) or not fixed_lines:
        fixed_lines.append(line)
    else:
        fixed_lines[-1] += " " + line  

print(f"{len(raw_lines)} raw lines -> {len(fixed_lines)} logical rows after stitching")


1020 raw lines -> 1003 logical rows after stitching


Some rows use `;` instead of `,`. 

In [9]:
def normalize_delims(line):
    if line.count(";") > line.count(","):
        return line.replace(";", ",")
    return line

fixed_lines = [normalize_delims(l) for l in fixed_lines]
header_line, *body_lines = [l for l in fixed_lines if l.strip() != ""]


De-duplicate the header

In [12]:
raw_header = next(csv.reader([header_line]))

seen = {}
header = []
for h in raw_header:
    if h in seen:
        seen[h] += 1
        header.append(f"{h}_{seen[h]}")
    else:
        seen[h] = 0
        header.append(h)

ncols = len(header)
print("Header:", header)


Header: ['', 'txn_id', 'txn_date', 'merchant', 'category', 'amount', 'amount_1', 'currency', 'status', 'card_last4', 'metadata', 'notes']


Parse leniently, drop unrecoverable rows

In [13]:
body_lines = [l.replace("\x00", "") for l in body_lines]  
reader = csv.reader(body_lines)
clean_rows, skipped = [], 0
for r in reader:
    if len(r) == ncols:
        clean_rows.append(r)
    else:
        skipped += 1

print(f"Skipped {skipped} structurally unrecoverable rows")

df = pd.DataFrame(clean_rows, columns=header)
df = df.drop(columns=[c for c in df.columns if c == ""])
df.shape

Skipped 23 structurally unrecoverable rows


(973, 11)

In [14]:
if "amount_1" in df.columns:
    df["amount"] = df["amount"].where(df["amount"].notna() & (df["amount"] != ""), df["amount_1"])
    df = df.drop(columns=["amount_1"])

df.columns.tolist()


['txn_id',
 'txn_date',
 'merchant',
 'category',
 'amount',
 'currency',
 'status',
 'card_last4',
 'metadata',
 'notes']

Normalize every "null" spelling

In [15]:
null_tokens = {
    "", "NA", "N/A", "null", "NULL", "None", "-", "missing", "n/a", "--", "#N/A", "??",
    "#DIV/0!", "#REF!", "#VALUE!", "#NAME?", "TBD", "unknown", "UNK", "0000-00-00", "  ",
    "N\\A", "nan", "NaN", "Infinity", "-Infinity", "undefined", "\\N", "(null)",
}

df = df.replace(list(null_tokens), np.nan)
df = df.dropna(how="all")  # drop rows that are now entirely empty
df.shape


(967, 10)

Strip symbol noise

In [16]:
def strip_junk(s):
    if pd.isna(s):
        return s
    s = str(s)
    s = s.replace("\ufffd", "").replace("\x00", "")   # decode-error / NUL artifacts
    s = re.sub(r"[\u200b\u200e\u200f]", "", s)          # zero-width characters
    s = re.sub(r"(###|@@@|\*\*\*|<<>>|\u201aÄî)", "", s)  # stray symbol tails
    s = re.sub(r"<[^>]+>", "", s)                        # HTML tags
    s = s.replace("&nbsp;", " ")
    s = s.replace('"weird"', "")                          # injected unescaped-quote artifact
    s = re.sub(r"\s+", " ", s).strip()
    return s if s else np.nan

SYMBOL_STRIP = re.compile(r"[#$%&*@~^\u00a7\u00b6\u2022\u2020\u00b1\u2248\u2026\u25c6\u25b2\u2021\u25ca\u00a4\u25cb!?/\[\]\(\)]+")

def strip_symbols(s):
    if pd.isna(s):
        return s
    s = SYMBOL_STRIP.sub(" ", str(s))  
    s = re.sub(r"\s+", " ", s).strip()
    return s if s else np.nan


for col in ["merchant", "notes", "category", "status", "txn_id", "currency", "card_last4"]:
    if col in df.columns:
        df[col] = df[col].apply(strip_junk)
        df[col] = df[col].apply(strip_symbols)


In [17]:
sql_pattern = re.compile(r"(DROP TABLE|SELECT \*|OR '1'='1)", re.I)
HOMOGLYPH_MAP = str.maketrans({"\u0405": "S", "\u0430": "a", "\u216d": "C", "\u2164": "V", "\uff34": "T"})
mojibake_pattern = re.compile(r"[\u00c3\u00c2\u00a1\u00fc\u00bb\u0178]")

for col in ["merchant", "notes"]:
    if col not in df.columns:
        continue
    df[col] = df[col].apply(lambda s: s.translate(HOMOGLYPH_MAP) if isinstance(s, str) else s)
    df[col] = df[col].astype(str).str.replace("\u00a0", " ", regex=False)
    df.loc[df[col].astype(str).str.contains(sql_pattern, na=False, regex=True), col] = np.nan
    df.loc[df[col].astype(str).str.contains(mojibake_pattern, na=False), col] = np.nan
    df.loc[df[col].astype(str).str.contains(r"[\u0300-\u036F]", na=False, regex=True), col] = np.nan  
    df.loc[df[col].astype(str).str.contains(r"[\u0400-\u04FF]", na=False, regex=True), col] = np.nan  
df.loc[df["notes"].astype(str).str.len() > 100, "notes"] = np.nan  # catches the fake-JWT blob
df = df.replace("nan", np.nan)


/var/folders/p6/qv1dvrpx05g5xp58t734m5jw0000gn/T/ipykernel_67306/336318853.py:10: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df.loc[df[col].astype(str).str.contains(sql_pattern, na=False, regex=True), col] = np.nan
/var/folders/p6/qv1dvrpx05g5xp58t734m5jw0000gn/T/ipykernel_67306/336318853.py:10: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df.loc[df[col].astype(str).str.contains(sql_pattern, na=False, regex=True), col] = np.nan


Dates

In [18]:
FULLWIDTH_BACK = str.maketrans("０１２３４５６７８９", "0123456789")

def parse_date(d):
    if pd.isna(d):
        return pd.NaT
    d = str(d).strip().lstrip("'").translate(FULLWIDTH_BACK)
    d = re.sub(r"\s*\+\d{2}:\d{2}$", "", d)  # strip UTC offset suffix
    if re.fullmatch(r"\d{8}", d):
        return pd.to_datetime(d, format="%Y%m%d", errors="coerce")
    if re.fullmatch(r"\d{2}\.\d{2}\.\d{4}", d):
        return pd.to_datetime(d, format="%d.%m.%Y", errors="coerce")
    return pd.to_datetime(d, errors="coerce", format="mixed")

if "txn_date" in df.columns:
    df["txn_date"] = df["txn_date"].apply(parse_date)

df["txn_date"].isna().sum(), df["txn_date"].head()


(np.int64(137),
 0    2023-02-20 00:00:00
 1    2021-04-10 00:00:00
 2                    NaT
 3    2022-12-26 00:00:00
 4    2024-09-10 00:00:00
 Name: txn_date, dtype: object)

Category and status

In [19]:
if "category" in df.columns:
    df["category"] = (df["category"].astype(str)
                       .str.replace("_", " ")
                       .str.rstrip("!")
                       .str.strip()
                       .str.title()
                       .replace("Nan", np.nan))

if "status" in df.columns:
    df["status"] = (df["status"].astype(str)
                     .str.strip("[]? ")
                     .str.title()
                     .replace("Nan", np.nan))


 Amount

Hardest column: accounting negatives `(301.50)`, sci notation, dual-currency mashups, European decimals, K/M abbreviations, literal `Infinity`/`NaN`/`-0`.

In [20]:
def fix_amount(v):
    if pd.isna(v):
        return np.nan
    v = str(v).strip().translate(FULLWIDTH_BACK)
    if v in {"Infinity", "-Infinity", "NaN"}:
        return np.nan

    # K/M magnitude suffix
    m = re.fullmatch(r"(-?\d+\.?\d*)([KM])", v)
    if m:
        mult = 1_000 if m.group(2) == "K" else 1_000_000
        return float(m.group(1)) * mult

    # dual-currency mashup  keep only the first figure
    if "(" in v and "≈" in v:
        v = v.split("(")[0].strip()

    negative = v.startswith("(") and v.endswith(")")
    v = v.strip("()~ \t")

    # European decimal format: 1.234,56 -> 1234.56
    if re.fullmatch(r"-?\d{1,3}(\.\d{3})*,\d{2}", v):
        v = v.replace(".", "").replace(",", ".")

    v = re.sub(r"[^\d.eE+-]", "", v)
    if v in ("", "-", ".", "-0", "0"):
        return 0.0 if v in ("-0", "0") else np.nan

    try:
        num = float(v)
    except ValueError:
        return np.nan
    return -abs(num) if negative else num

if "amount" in df.columns:
    df["amount"] = df["amount"].apply(fix_amount)

df["amount"].describe()


count    8.920000e+02
mean     1.261122e+06
std      7.763789e+06
min     -9.369531e+05
25%      1.010380e+05
50%      4.004249e+05
75%      6.819149e+05
max      9.860000e+07
Name: amount, dtype: float64

Currency

Collapse every spelling to an ISO code.

In [21]:
currency_map = {"usd": "USD", "eur": "EUR", "dollars": "USD", "$": "USD",
                "€": "EUR", "gbp": "GBP", "jpy": "JPY"}

if "currency" in df.columns:
    df["currency"] = df["currency"].astype(str).str.strip()
    df["currency"] = df["currency"].apply(
        lambda c: np.nan if (pd.isna(c) or c == "nan")
        else currency_map.get(str(c).lower(), c if str(c).upper() in {"USD", "EUR", "GBP", "JPY"} else np.nan)
    )

df["currency"].value_counts(dropna=False)


currency
EUR    302
USD    270
NaN    219
GBP     89
JPY     87
Name: count, dtype: int64

 Repair broken JSON metadata

In [22]:
def parse_metadata(m):
    if pd.isna(m):
        return {}
    m = str(m).strip()
    if m.startswith("["):
        m = m[1:].split("},")[0] + "}"       # take just the first object out of the array
    m = m.split("}{")[0] + ("}" if "}{" in m else "")  # split two glued-together objects

    for attempt in (m, m.replace("=", ":")):
        try:
            return json.loads(attempt)
        except json.JSONDecodeError:
            pass
        
        py_attempt = re.sub(r"\btrue\b", "True", attempt)
        py_attempt = re.sub(r"\bfalse\b", "False", py_attempt)
        py_attempt = re.sub(r"\bnull\b", "None", py_attempt)
        try:
            return ast.literal_eval(py_attempt)
        except (ValueError, SyntaxError):
            pass

    repaired = re.sub(r"(\w+)\s*[:=]", r'"\1":', m)      
    if repaired.count("{") > repaired.count("}"):
        repaired += "}" * (repaired.count("{") - repaired.count("}"))
    try:
        return json.loads(repaired)
    except json.JSONDecodeError:
        return {}

if "metadata" in df.columns:
    meta = df["metadata"].apply(parse_metadata)
    df["meta_channel"] = meta.apply(lambda d: d.get("channel"))
    df["meta_fee"] = meta.apply(lambda d: d.get("fee"))
    df["meta_loyalty_pts"] = meta.apply(lambda d: d.get("loyalty_pts"))
    df["meta_flagged"] = meta.apply(lambda d: d.get("flagged"))
    df = df.drop(columns=["metadata"])

df[["meta_channel", "meta_fee", "meta_loyalty_pts", "meta_flagged"]].head()


,meta_channel,meta_fee,meta_loyalty_pts,meta_flagged
0,phone,3.07,303.0,True
1,phone,0.27,371.0,True
2,mobile,0.44,425.0,False
3,online,0.82,155.0,False
4,in-store,2.64,3.0,True


`card_last4`

In [23]:
if "card_last4" in df.columns:
    df["card_last4"] = pd.to_numeric(df["card_last4"], errors="coerce")
    df.loc[(df["card_last4"] < 0) | (df["card_last4"] > 9999), "card_last4"] = np.nan


Deduplicate, flag conflicts

In [24]:
df["txn_id"] = (df["txn_id"].astype(str).str.strip().str.upper()
                 .str.replace("TXN_", "TXN-")
                 .str.replace("-R", "", regex=False))
df = df[df["txn_id"] != "NAN"]

conflict_mask = df.duplicated(subset=["txn_id"], keep=False) & df["txn_id"].notna()
if conflict_mask.any():
    print(f"{conflict_mask.sum()} rows share a txn_id with a DIFFERING amount -- flagged, not auto-resolved:")
    display(df.loc[conflict_mask, ["txn_id", "amount"]].sort_values("txn_id"))

df = df.drop_duplicates(subset=["txn_id"], keep="first")

zero_mask = df["amount"] == 0
if zero_mask.any():
    print(f"\n{zero_mask.sum()} rows have a literal $0.00 amount -- flagged for review, not treated as normal:")
    display(df.loc[zero_mask, ["txn_id", "merchant", "amount"]].head(10))

df = df.reset_index(drop=True)
df.shape


74 rows share a txn_id with a DIFFERING amount -- flagged, not auto-resolved:


,txn_id,amount
773,TXN-50078,-411996.37
82,TXN-50078,-411996.37
447,TXN-50094,944172.05
32,TXN-50094,944172.05
148,TXN-50101,NaN
...,...,...
259,TXN50835,96790000.00
913,TXN50850,674289.47
601,TXN50850,674289.47
758,TXN50905,559643.67



49 rows have a literal $0.00 amount -- flagged for review, not treated as normal:


,txn_id,merchant,amount
7,TXN50679,Chipotle,0.0
22,TXN-50342,Chipotle,0.0
30,TXN-50866,Apple Store,0.0
47,TXN50105,Target,0.0
59,TXN50516,CVS Pharmacy,0.0
79,TXN-50557,void,0.0
91,TXN-50763,Netflix,0.0
136,TXN50766,Apple Store,0.0
139,TXN-50476,Chipotle,0.0
153,TXN-50906,"Amazon said ""call me back""",0.0


(898, 13)

Impute remaining nulls
 Strategy per column:
- `txn_id`  not imputed, row dropped 
- `txn_date`  median
- categorical fields  `"Unknown"`
- `currency`  mode
- `notes`  `"No notes provided"`
- numeric fields  median (robust to outliers)
- `meta_flagged`  mode


In [25]:
imputed_flags = pd.DataFrame(index=df.index)

before_n = len(df)
df = df[df["txn_id"].notna()].reset_index(drop=True)
imputed_flags = imputed_flags.reindex(df.index)
print(f"Dropped {before_n - len(df)} row(s) with no txn_id (unrecoverable, not imputed)")

# txn_date: median date
imputed_flags["txn_date"] = df["txn_date"].isna()
parsed_dates = pd.to_datetime(df["txn_date"], errors="coerce", utc=True).dt.tz_localize(None)
median_date = parsed_dates.dropna().median()
df["txn_date"] = parsed_dates.fillna(median_date)

# categorical / text columns
for col, fill in [("merchant", "Unknown"), ("category", "Unknown"),
                   ("currency", df["currency"].mode(dropna=True).iloc[0] if not df["currency"].mode(dropna=True).empty else "USD"),
                   ("status", "Unknown"), ("notes", "No notes provided"),
                   ("meta_channel", "Unknown")]:
    if col in df.columns:
        imputed_flags[col] = df[col].isna()
        df[col] = df[col].fillna(fill)

# numeric columns: median
for col in ["amount", "card_last4", "meta_fee", "meta_loyalty_pts"]:
    if col in df.columns:
        imputed_flags[col] = df[col].isna()
        df[col] = df[col].fillna(df[col].median())

# meta_flagged: mode
if "meta_flagged" in df.columns:
    imputed_flags["meta_flagged"] = df["meta_flagged"].isna()
    mode_val = df["meta_flagged"].mode(dropna=True)
    df["meta_flagged"] = df["meta_flagged"].fillna(mode_val.iloc[0] if not mode_val.empty else False)

print("\nImputed value counts per column:")
print(imputed_flags.sum())
print(f"\nTotal cells imputed: {int(imputed_flags.sum().sum())} out of {df.shape[0] * df.shape[1]:,} cells")

df.isna().sum()


Dropped 0 row(s) with no txn_id (unrecoverable, not imputed)

Imputed value counts per column:
txn_date            122
merchant             58
category             55
currency            203
status               34
notes               156
meta_channel        207
amount               70
card_last4           66
meta_fee            207
meta_loyalty_pts    207
meta_flagged        207
dtype: int64

Total cells imputed: 1592 out of 11,674 cells


/var/folders/p6/qv1dvrpx05g5xp58t734m5jw0000gn/T/ipykernel_67306/4011517574.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["meta_flagged"] = df["meta_flagged"].fillna(mode_val.iloc[0] if not mode_val.empty else False)


txn_id              0
txn_date            0
merchant            0
category            0
amount              0
currency            0
status              0
card_last4          0
notes               0
meta_channel        0
meta_fee            0
meta_loyalty_pts    0
meta_flagged        0
dtype: int64

Save and review

In [26]:
df.to_csv(CLEAN_FILE, index=False)
print(f"Saved {len(df):,} clean rows to {CLEAN_FILE}")
df.head(10)


Saved 898 clean rows to clean_confusion_financial_transactions.csv


,txn_id,txn_date,merchant,category,amount,currency,status,card_last4,notes,meta_channel,meta_fee,meta_loyalty_pts,meta_flagged
0,TXN50456,2023-02-20 00:00:00,Unknown,Unknown,94377.84,USD,Refunded,8942.0,Recurring payment,phone,3.07,303.0,True
1,TXN50675,2021-04-10 00:00:00,Apple Store,Healthcare,-262519.00,EUR,Failed,7179.0,No notes provided,phone,0.27,371.0,True
2,TXN-50608,2023-07-15 12:00:00,Target,Transport,497759.78,GBP,Failed,8549.0,customer disputed charge,mobile,0.44,425.0,False
3,TXN-50583,2022-12-26 00:00:00,Airbnb,Subscription,36942.72,EUR,Unknown,2294.0,customer disputed charge,online,0.82,155.0,False
4,TXN-50310,2024-09-10 00:00:00,Spotify,Subscription,733902.31,GBP,Pending,9537.0,No notes provided,in-store,2.64,3.0,True
5,TXN50376,2025-06-04 00:00:00,Uber,Dining,393860.20,EUR,Completed,5609.0,"call back re: acct 88213, ref: 'urgent '",mobile,1.64,140.0,False
6,TXN-50150,2021-03-09 22:09:00,Target,Transport,373300.00,EUR,Refunded,9059.0,amazon,in-store,1.76,460.0,True
7,TXN50679,2023-07-15 12:00:00,Chipotle,Transport,0.00,EUR,Unknown,6227.0,"call back re: acct 88213, ref: 'urgent '",mobile,4.44,345.0,True
8,TXN-50468,2020-01-16 00:00:00,Uber,Healthcare,410000.00,EUR,Refunded,2231.0,"call back re: acct 88213, ref: 'urgent '",online,4.85,84.0,True
9,TXN50213,2024-08-21 00:00:00,Target,Healthcare,618800.00,EUR,Pending,6758.0,No notes provided,mobile,0.72,238.0,True


In [23]:
print("Remaining nulls per column (these are now REAL nulls, not disguised ones):")
df.isna().sum()


Remaining nulls per column (these are now REAL nulls, not disguised ones):


txn_id              0
txn_date            0
merchant            0
category            0
amount              0
currency            0
status              0
card_last4          0
notes               0
meta_channel        0
meta_fee            0
meta_loyalty_pts    0
meta_flagged        0
dtype: int64